# Kaggle GPU runner (thin wrapper)

Runs foundation-model inference and the deep baselines for `tsfm-grid-calibration`.

Settings: **Accelerator = GPU (T4 x2 or P100)**, **Internet = On**, attach the private dataset holding
`data/clean/*.parquet` (and `configs/tuned/` if needed). Secrets (Add-ons -> Secrets): `GITHUB_TOKEN`
(fine-grained, read-only, for a private repo) and `HF_TOKEN`.

For long runs use **Save Version -> Save & Run All (Commit)**; outputs in `/kaggle/working` are kept.

In [ ]:
# ---- parameters ----
REPO = "TanishqX-404/tsfm-grid-calibration"
BRANCH = "main"
DATASET_DIR = "/kaggle/input/tgc-clean"      # attached private dataset with data/clean/*.parquet
MODELS = ["chronos2", "timesfm25", "tirex", "moirai2"]  # foundation models
NEURAL = ["nhits", "patchtst"]               # deep baselines (3 seeds each)
PHASE = "val"                                # "val" before the freeze; "test" only after tagging v-frozen
DRY_RUN = True                               # first run: 3 days per series; then set False

In [ ]:
import os, subprocess
from kaggle_secrets import UserSecretsClient
sec = UserSecretsClient()
try:
    os.environ["HF_TOKEN"] = sec.get_secret("HF_TOKEN")
except Exception:
    print("no HF_TOKEN secret (anonymous downloads may be rate-limited)")
try:
    tok = sec.get_secret("GITHUB_TOKEN")
    url = f"https://{tok}@github.com/{REPO}.git"
except Exception:
    url = f"https://github.com/{REPO}.git"
if not os.path.exists("/kaggle/working/tgc"):
    subprocess.run(["git", "clone", "-b", BRANCH, "--depth", "1", url, "/kaggle/working/tgc"], check=True)
os.chdir("/kaggle/working/tgc")
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

In [ ]:
# Do not reinstall torch/numpy: keep Kaggle's preinstalled versions.
!pip install -q -r requirements-kaggle.txt
!pip install -q --no-deps "git+https://github.com/SalesforceAIResearch/uni2ts.git"
!pip install -q --no-deps -e .

In [ ]:
!nvidia-smi
import torch, importlib
print("torch", torch.__version__, "cuda", torch.version.cuda, torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
for p in ["chronos", "timesfm", "tirex", "uni2ts", "neuralforecast", "pandas", "numpy"]:
    try:
        print(p, getattr(importlib.import_module(p), "__version__", "installed"))
    except Exception as e:
        print(p, "IMPORT FAILED:", e)

In [ ]:
# Link the prepared data into the repo layout
os.makedirs("data", exist_ok=True)
if not os.path.exists("data/clean"):
    src = os.path.join(DATASET_DIR, "clean") if os.path.isdir(os.path.join(DATASET_DIR, "clean")) else DATASET_DIR
    os.symlink(src, "data/clean")
print(os.listdir("data/clean"))

In [ ]:
import sys
allow = ["--allow-test"] if PHASE != "val" else []
dry = ["--dry-run"] if DRY_RUN else []
failed = []
for m in MODELS:
    r = subprocess.run([sys.executable, "-m", "tgc.forecast.run", "--model", m, "--phase", PHASE, *allow, *dry])
    if r.returncode: failed.append(m)
for m in NEURAL:
    for s in (0, 1, 2):
        r = subprocess.run([sys.executable, "-m", "tgc.forecast.run", "--model", m, "--seed", str(s), "--phase", PHASE, *allow, *dry])
        if r.returncode: failed.append(f"{m}-s{s}")
print("failed:", failed)

In [ ]:
# M4 acceptance: identical outputs on two runs (dry run only)
if DRY_RUN:
    for m in MODELS:
        subprocess.run([sys.executable, "-m", "tgc.forecast.determinism", "--model", m])

In [ ]:
# Package outputs for download (or publish them as a private Kaggle dataset)
!cd /kaggle/working/tgc && zip -qr /kaggle/working/forecasts_{PHASE}{'_dry' if DRY_RUN else ''}.zip out/forecasts out/dry_run out/env 2>/dev/null; ls -la /kaggle/working